In [1]:
import numpy as np
import matplotlib.pyplot as plt


échauffement: le broadcasting

In [4]:
a=np.array([1,2,3])
print(a)

[1 2 3]


In [3]:
#1. On cherche la forme
print(a[:,None])

[[1]
 [2]
 [3]]


On a ajouté une dimension: on passe de (3,) à (3,1) en "créant" une colonne 

In [ ]:
# Puis n cherche la forme
print(a[None,:])

[[1 2 3]]


Cette fois-ci, on passe de (3,) à (1,3)

In [8]:
#2.
print(a[:,None]-a[None,:])

[[ 0 -1 -2]
 [ 1  0 -1]
 [ 2  1  0]]


On a "dupliqué" la colonne de a[:,None] pour avoir une matrice de taille 3*3, et à chacune des lignes on a enlevé la matrice ligne a[None,:]: on a donc une forme (3,3) et l'élément [i,j] contient i-j

In [6]:
#3. je me donne un tableau b qui est de taille (2,3)
b=np.array([[1,2,3],[4,5,6]])
print(b)

[[1 2 3]
 [4 5 6]]


In [10]:
#Je décompose par étapes
print(b[:,None,:])

[[[1 2 3]]

 [[4 5 6]]]


La taille est (2,1,3)

In [11]:
print(b[:,:,None])

[[[1]
  [2]
  [3]]

 [[4]
  [5]
  [6]]]


La taille est (2,3,1)

In [16]:
c=b[:,None,:]-b[:,:,None]
print(c)

[[[ 0  1  2]
  [-1  0  1]
  [-2 -1  0]]

 [[ 0  1  2]
  [-1  0  1]
  [-2 -1  0]]]


initialisation aléatoire

In [15]:
# les bornes pour le tirage au sort initial
mass_max = 3.
    
x_min, x_max = -10., 10.
y_min, y_max = -10., 10.

speed_max = 1.


In [58]:
def init_problem(N):
    masses=np.random.uniform(0.001,mass_max,(N,))
    low=np.array([[x_min],[y_min]])
    high=np.array([[x_max],[y_max]])
    positions=np.random.uniform(low,high,(2,N))
    speeds=np.random.uniform(0,speed_max,(2,N))
    return masses,positions, speeds, len(positions[0])

In [36]:
# pour tester

# normalement vous devez pouvoir faire ceci

masses, positions, speeds = init_problem(10)

# et ceci devrait afficher OK
try:
    assert masses.shape == (10,) and positions.shape == speeds.shape == (2, 10)
    print("OK")
except:
    print("KO")

OK


initialisation reproductible

In [37]:
# for your convenience

def init3():
    # first element is sun-like: heavy, at the center, and no speed
    masses = np.array([3, 1, 1], dtype=float)
    positions = np.array([
        [0, 5, -5], 
        [0, 1, -1]], dtype=float)
    speeds = np.array([
        [0, -1, 1], 
        [0, 0, 0]], dtype=float)
    return masses, positions, speeds


In [78]:
#Puisque l'on procède par étapes,
#je vais écrire plusieurs fonctions
#1.
def vecteurs(positions):
    diff=positions[:,None,:]-positions[:,:,None]
    return diff

In [52]:
#2. 
def distances(positions):
    dist=np.linalg.norm(vecteurs(positions),axis=0)
    return dist

dist[i,i] renvoit évidemment 0, puisqu'il ne faut pas diviser par 0 on va le remplacer par l'infini (contribution nulle)

In [56]:
def distancesv2(positions):
    dist=np.linalg.norm(vecteurs(positions),axis=0)
    np.fill_diagonal(dist,np.inf)
    return dist  

In [63]:
def prod_masses(masses):
    produit=masses[:,None]*masses[None,:]
    return produit


In [88]:
#3.
def coeff(positions,masses,G):
    return G*prod_masses(masses)/(distancesv2(positions)**3)

In [98]:
def coeff_final(masses,positions,G):
    return vecteurs(positions)*coeff(positions,masses,G)[None,:,:]

In [99]:
#4. La somme finale
def forces(masses,positions,G=1.0):
    return np.sum(coeff_final(masses,positions,G),axis=2)

In [102]:
# pour tester, voici les valeurs attendues avec la config prédéfinie

masses, positions, speeds = init3()

f = forces(masses, positions)

# should be true
np.all(np.isclose(f, np.array([
     [ 0.        , -0.12257258,  0.12257258],
     [ 0.        , -0.02451452,  0.02451452]])))

np.True_

le simulateur

In [ ]:
#cette fois-ci, on procède avec une unique fonction
